# Adım 1: Docker Ortamının Kurulumu

Bu notebook, NYC Taxi veri pipeline projesinin ilk adımını kapsamaktadır.  
Hedef: Tüm bileşenleri Docker konteynerleri içinde ayağa kaldırmak.

---

## Servisler

| Servis | Image | Port | Açıklama |
|--------|-------|------|----------|
| Zookeeper | confluentinc/cp-zookeeper:7.6.1 | 2181 | Kafka koordinasyon servisi |
| Kafka Broker | confluentinc/cp-kafka:7.6.1 | 9092, 29092 | Mesaj kuyruğu |
| Spark | bitnami/spark:3.5 (custom) | 8080, 7077 | Stream işleme motoru |
| MLflow | python:3.11-slim (custom) | 5000 | Model takip servisi |
| Producer | python:3.11-slim (custom) | - | Kafka'ya veri gönderici |

## 1. Ön Gereksinimler

Sisteminizde aşağıdakilerin kurulu olduğunu doğrulayın:

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = result.stdout.strip() or result.stderr.strip()
    print(output)
    return result.returncode

print("=== Docker Versiyonu ===")
run("docker --version")

print("\n=== Docker Compose Versiyonu ===")
run("docker compose version")

print("\n=== Docker Daemon Durumu ===")
run("docker info --format 'Server Version: {{.ServerVersion}}'")

## 2. Proje Dizin Yapısı

Pipeline'ın beklediği dosya/klasör yapısını inceleyelim:

In [ ]:
import os
from pathlib import Path

# Notebook'un bulunduğu konumdan proje köküne git
PROJECT_ROOT = Path(os.getcwd()).parent
print(f"Proje Kök Dizini: {PROJECT_ROOT}\n")

kritik_dosyalar = [
    "docker-compose.yml",
    "producer/Dockerfile",
    "producer/main.py",
    "producer/requirements.txt",
    "spark_jobs/Dockerfile",
    "spark_jobs/requirements.txt",
    ".env.example",
]

print("Kritik Dosyaların Varlık Kontrolü:")
print("-" * 45)
all_ok = True
for dosya in kritik_dosyalar:
    yol = PROJECT_ROOT / dosya
    durum = "✅ VAR" if yol.exists() else "❌ YOK"
    if not yol.exists():
        all_ok = False
    print(f"{durum}  {dosya}")

print("-" * 45)
print("Tüm dosyalar mevcut!" if all_ok else "⚠️  Bazı dosyalar eksik!")

## 3. docker-compose.yml İncelenmesi

In [ ]:
compose_path = PROJECT_ROOT / "docker-compose.yml"

with open(compose_path, "r", encoding="utf-8") as f:
    icerik = f.read()

print(icerik)

## 4. Dockerfile'ların İncelenmesi

In [ ]:
dockerfiles = [
    ("Producer Dockerfile", PROJECT_ROOT / "producer" / "Dockerfile"),
    ("Spark Dockerfile",    PROJECT_ROOT / "spark_jobs" / "Dockerfile"),
]

for baslik, yol in dockerfiles:
    print(f"{'='*50}")
    print(f"  {baslik}")
    print(f"  Konum: {yol}")
    print(f"{'='*50}")
    with open(yol, "r", encoding="utf-8") as f:
        print(f.read())
    print()

## 5. .env Dosyasının Hazırlanması

`.env.example` dosyasını `.env` olarak kopyalayıp gerekli değerleri ayarlayın:

In [ ]:
import shutil

env_example = PROJECT_ROOT / ".env.example"
env_file    = PROJECT_ROOT / ".env"

print("=== .env.example İçeriği ===")
with open(env_example, "r", encoding="utf-8") as f:
    print(f.read())

if not env_file.exists():
    shutil.copy(env_example, env_file)
    print("\n✅ .env dosyası oluşturuldu.")
else:
    print("\nℹ️  .env dosyası zaten mevcut, değiştirilmedi.")

## 6. Docker Image'larının Build Edilmesi

Özel Dockerfile'ları olan `producer` ve `spark` servislerini build ediyoruz:

In [ ]:
import subprocess, os

os.chdir(PROJECT_ROOT)
print(f"Çalışma dizini: {os.getcwd()}\n")

print("=" * 50)
print("Docker image'ları build ediliyor... (birkaç dakika sürebilir)")
print("=" * 50)

result = subprocess.run(
    "docker compose build --no-cache",
    shell=True, capture_output=False, text=True
)

if result.returncode == 0:
    print("\n✅ Build başarıyla tamamlandı!")
else:
    print("\n❌ Build sırasında hata oluştu. Yukarıdaki çıktıyı inceleyin.")

## 7. Servislerin Ayağa Kaldırılması

Temel servisleri (Zookeeper, Kafka, Spark, MLflow) başlatıyoruz:

In [ ]:
print("Servisler başlatılıyor...")

result = subprocess.run(
    "docker compose up -d",
    shell=True, capture_output=True, text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("\n✅ Servisler başlatıldı!")
else:
    print("\n❌ Başlatma sırasında hata oluştu.")

## 8. Servis Durumlarının Doğrulanması

Her servisin `running` ve `healthy` olduğunu kontrol ediyoruz:

In [ ]:
import time, json

print("Servislerin hazır olması için 20 saniye bekleniyor...")
time.sleep(20)

result = subprocess.run(
    'docker compose ps --format json',
    shell=True, capture_output=True, text=True
)

print(f"{'Servis Adı':<30} {'Durum':<15} {'Port':<20}")
print("-" * 65)

for satir in result.stdout.strip().splitlines():
    try:
        s = json.loads(satir)
        ad     = s.get("Name", s.get("Service", "?"))
        durum  = s.get("State", s.get("Status", "?"))
        portlar = s.get("Publishers", [])
        port_str = ", ".join(
            f"{p.get('PublishedPort','')}:{p.get('TargetPort','')}"
            for p in portlar if p.get("PublishedPort")
        ) or "-"
        ikon = "✅" if durum == "running" else "❌"
        print(f"{ikon} {ad:<28} {durum:<15} {port_str:<20}")
    except:
        print(satir)

## 9. Kafka Topic Doğrulaması

Kafka broker'ın topic listeleyebildiğini test ediyoruz:

In [ ]:
print("=== Kafka Topic Listesi ===")
result = subprocess.run(
    "docker exec nyc-taxi-kafka kafka-topics "
    "--bootstrap-server localhost:9092 --list",
    shell=True, capture_output=True, text=True
)
output = result.stdout.strip()
print(output if output else "(Henüz topic yok — bu normaldir)")

if result.returncode == 0:
    print("\n✅ Kafka broker erişilebilir durumda!")
else:
    print("\n❌ Kafka'ya bağlanılamadı:")
    print(result.stderr)

## 10. Spark UI Erişim Kontrolü

In [ ]:
import urllib.request

kontroller = [
    ("Spark Master UI",  "http://localhost:8080"),
    ("MLflow UI",        "http://localhost:5000"),
]

print(f"{'Servis':<20} {'URL':<30} {'Durum'}")
print("-" * 60)
for ad, url in kontroller:
    try:
        urllib.request.urlopen(url, timeout=5)
        print(f"✅ {ad:<18} {url:<30} Erişilebilir")
    except Exception as e:
        print(f"❌ {ad:<18} {url:<30} {type(e).__name__}")

## 11. Servislerin Durdurulması

Testi tamamladıktan sonra servisleri durdurmak için bu hücreyi çalıştırın:

In [ ]:
# Bu hücreyi yalnızca servisleri durdurmak istediğinizde çalıştırın!
# result = subprocess.run("docker compose down", shell=True, capture_output=True, text=True)
# print(result.stdout)
# print("✅ Tüm servisler durduruldu.")

print("ℹ️  Servisleri durdurmak için bu hücredeki yorum satırlarını kaldırın.")

## 12. Özet & Teslim Edilecekler

Bu adımda tamamlananlar:

| # | Teslim | Durum |
|---|--------|-------|
| 1 | `docker-compose.yml` (Zookeeper + Kafka + Spark + MLflow + Producer) | ✅ |
| 2 | `producer/Dockerfile` — Python Producer konteyneri | ✅ |
| 3 | `spark_jobs/Dockerfile` — Apache Spark çalışma ortamı | ✅ |
| 4 | Servislerin ayağa kalktığını gösteren doğrulama | ✅ (Bölüm 8) |

### Servis Erişim Adresleri

| Servis | URL |
|--------|-----|
| Spark Master UI | http://localhost:8080 |
| MLflow UI | http://localhost:5000 |
| Kafka Broker (dış) | localhost:29092 |
| Zookeeper | localhost:2181 |

---
> **Sonraki adım →** Adım 2: Kafka Producer ile veri akışının başlatılması